In [68]:
from sklearn.model_selection import train_test_split
import os
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

In [2]:
os.listdir('data/')
noise_data = 'data/dataCNNdf/test/'
ver_data_crawl = 'data/dataCNNdf/verified_data_crawl/'
ver_data = 'data/dataCNNdf/verified_data/'

video_paths = []
for folder in [noise_data, ver_data_crawl, ver_data]:
    video_paths.extend([os.path.join(folder, f) for f in os.listdir(folder)])

len(video_paths)

1571

In [3]:
paths_train, paths_test = train_test_split(video_paths, test_size=0.2, random_state=25)

len(paths_train), len(paths_test)

(1256, 315)

In [4]:
import torch

In [5]:
tensors = [torch.load(f) for f in tqdm(paths_train)]
full_tensor = torch.cat(tensors, dim=0)

print(full_tensor.shape)

100%|██████████████████████████████████████████████████████████████████████████████| 1256/1256 [00:13<00:00, 95.48it/s]

torch.Size([10651, 3, 10, 33])


In [8]:
labels_list = []

for f in paths_train:
    video_label = f.split('/')[-1].split('-')[0]
    segments = torch.load(f)
    n_segments = segments.shape[0]
    labels_list.extend([video_label for _ in range(n_segments)])

In [11]:
len(labels_list)

10651

In [13]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train = le.fit_transform(labels_list)
y_train, len(y_train)

(array([7, 7, 7, ..., 9, 9, 9], dtype=int64), 10651)

In [15]:
isnan = torch.isnan(full_tensor).any().item()
isnan

False

In [16]:
tensors_test = [torch.load(f) for f in tqdm(paths_test)]
full_tensor_test = torch.cat(tensors_test, dim=0)

print(full_tensor_test.shape)

100%|████████████████████████████████████████████████████████████████████████████████| 315/315 [00:03<00:00, 88.02it/s]

torch.Size([2757, 3, 10, 33])


In [17]:
labels_list_test = []

for f in paths_test:
    video_label = f.split('/')[-1].split('-')[0]
    segments = torch.load(f)
    n_segments = segments.shape[0]
    labels_list_test.extend([video_label for _ in range(n_segments)])

In [19]:
len(labels_list_test)

2757

In [20]:
y_test = le.fit_transform(labels_list_test)
y_test, len(y_test)

(array([4, 4, 4, ..., 9, 9, 9], dtype=int64), 2757)

In [23]:
isnan = torch.isnan(full_tensor_test).any().item()
isnan

False

In [25]:
#removing face landmarks
all_landmarks = list(range(33))

drop_indices = [
    1, 2, 3,    # left_eye_inner, left_eye, left_eye_outer
    4, 5, 6,    # right_eye_inner, right_eye, right_eye_outer
    9, 10,      # mouth_left, mouth_right
]

keep_indices = [i for i in all_landmarks if i not in drop_indices]
keep_indices_tensor = torch.tensor(keep_indices)
full_tensor_filtered = full_tensor[:, :, :, keep_indices_tensor]
full_tensor_test_filtered = full_tensor_test[:, :, :, keep_indices_tensor]

full_tensor_filtered.shape, full_tensor_test_filtered.shape

(torch.Size([10651, 3, 10, 25]), torch.Size([2757, 3, 10, 25]))

In [22]:
#Model

In [28]:
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.backends.cudnn as cudnn
from torch.utils.data import DataLoader, TensorDataset

from tqdm import tqdm_notebook

import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(device)

cuda


In [36]:
y_tens_train = torch.tensor(y_train)
y_tens_test = torch.tensor(y_test)

y_tens_train.shape, y_tens_test.shape 

(torch.Size([10651]), torch.Size([2757]))

In [38]:
batch_size = 64
train_dataset = TensorDataset(full_tensor_filtered, y_tens_train)
test_dataset = TensorDataset(full_tensor_test_filtered, y_tens_test)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, 
                         num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,
                        num_workers=4, pin_memory=True)

In [57]:
class Depthwise1x1Conv(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=3):
        super().__init__()
        self.depthwise = nn.Conv2d(
            in_channels, in_channels, kernel_size,
            stride=stride, padding=padding, groups=in_channels, bias=False
        )
        self.bn1 = nn.BatchNorm2d(in_channels)
        self.pointwise = nn.Conv2d(in_channels, out_channels, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
    
    def forward(self, x):
        x = self.depthwise(x)
        x = F.relu(self.bn1(x))
        x = self.pointwise(x)
        x = F.relu(self.bn2(x))
        return x

class MultiScaleTemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.branch1 = nn.Conv2d(in_channels, out_channels//3, kernel_size=3, 
                                 padding=1, dilation=1)  # local
        self.branch2 = nn.Conv2d(in_channels, out_channels//3, kernel_size=3, 
                                 padding=2, dilation=2)  # medium
        self.branch3 = nn.Conv2d(in_channels, out_channels//3, kernel_size=3, 
                                 padding=3, dilation=3)  # long
        
        self.bn1 = nn.BatchNorm2d(out_channels//3)
        self.bn2 = nn.BatchNorm2d(out_channels//3)
        self.bn3 = nn.BatchNorm2d(out_channels//3)
        
        self.mix = nn.Conv2d(out_channels, out_channels, 1)
        self.bn_mix = nn.BatchNorm2d(out_channels)
        
    def forward(self, x):
        b1 = F.relu(self.bn1(self.branch1(x)))
        b2 = F.relu(self.bn2(self.branch2(x)))
        b3 = F.relu(self.bn3(self.branch3(x)))
        
        x = torch.cat([b1, b2, b3], dim=1)
        x = self.mix(x)
        return F.relu(self.bn_mix(x))

In [74]:
class PoseClassifier(nn.Module):
    def __init__(self, n_landmarks=22, n_coords=3, seq_length=10, 
                 num_classes=22, dropout_rate=0.2):
        super().__init__()
        
        #first conv 3x3 with concat
        self.stem = MultiScaleTemporalBlock(3, 9)
        
        # Depthwise blocks
        self.block1 = Depthwise1x1Conv(9, 32, stride=1)
        self.block2 = Depthwise1x1Conv(32, 64, stride=2)
        self.block3 = Depthwise1x1Conv(64, 128, stride=2)
        self.block4 = Depthwise1x1Conv(128, 256, stride=2)
        self.block5 = Depthwise1x1Conv(256, 256, stride=2)
        
        # BatchNorm after residual
        self.bn2 = nn.BatchNorm2d(64)
        self.bn3= nn.BatchNorm2d(128)
        self.bn4 = nn.BatchNorm2d(256)
        
        # Spatial Dropout
        self.spatial_dropout1 = nn.Dropout2d(dropout_rate)
        self.spatial_dropout2 = nn.Dropout2d(dropout_rate)
        
        # Global pooling
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        
        # Linear
        self.dropout = nn.Dropout(dropout_rate)
        self.fc = nn.Linear(256, num_classes)
        
        self._init_weights()
        
    def forward(self, x):
        # x: (batch, 3, time, landmarks)
        
        # Stem
        x = self.stem(x)  # (batch, 9, time, landmarks)
        x = self.spatial_dropout1(x)
        # Blocks
        x = self.block1(x) # (batch, 32, time, landmarks)
        x = self.block2(x) # 64
        x = self.block3(x) # 128
        x = self.block4(x) # 256
        x = self.block5(x) # 256
        
        # Spatial dropout
        x = self.spatial_dropout2(x)
        
        # Global pooling
        x = self.global_pool(x)
        x = x.view(x.size(0), -1)
        
        # Linear
        x = self.dropout(x)
        x = self.fc(x)
        
        return x
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

In [75]:
def train_model(model, train_loader, val_loader, epochs=100, lr=3e-4, model_save='model/CNN_model.pth'):
    
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': []
    }
    
    for epoch in tqdm(range(epochs)):
        # Training
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0
        
        train_bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs} [Train]')
        for inputs, labels in train_bar:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()
            
            train_bar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'acc': f'{100.*train_correct/train_total:.2f}%'
            })
        
        train_acc = 100. * train_correct / train_total
        train_loss = train_loss / len(train_loader)
        
        # Validation
        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            val_bar = tqdm(val_loader, desc=f'Epoch {epoch+1}/{epochs} [Val]')
            for inputs, labels in val_bar:
                inputs, labels = inputs.to(device), labels.to(device)
                
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()
                
                val_bar.set_postfix({
                    'loss': f'{loss.item():.4f}',
                    'acc': f'{100.*val_correct/val_total:.2f}%'
                })
        
        val_acc = 100. * val_correct / val_total
        val_loss = val_loss / len(val_loader)
        
        # Scheduler step
        scheduler.step()
        
        # History
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        print(f'\nEpoch {epoch+1}:')
        print(f'  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%')
        print(f'  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%')
        print(f'  LR: {optimizer.param_groups[0]["lr"]:.6f}')
    
    return history

In [70]:
model = PoseClassifier(
    n_landmarks=25,
    n_coords=3,
    seq_length=10,
    num_classes=22,
    dropout_rate=0.2
)

In [71]:
import torchsummary

torchsummary.summary(model.cuda(), (3, 10, 25))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1            [-1, 3, 10, 25]              84
       BatchNorm2d-2            [-1, 3, 10, 25]               6
            Conv2d-3            [-1, 3, 10, 25]              84
       BatchNorm2d-4            [-1, 3, 10, 25]               6
            Conv2d-5            [-1, 3, 10, 25]              84
       BatchNorm2d-6            [-1, 3, 10, 25]               6
            Conv2d-7            [-1, 9, 10, 25]              90
       BatchNorm2d-8            [-1, 9, 10, 25]              18
MultiScaleTemporalBlock-9            [-1, 9, 10, 25]               0
           Conv2d-10            [-1, 9, 14, 29]              81
      BatchNorm2d-11            [-1, 9, 14, 29]              18
           Conv2d-12           [-1, 32, 14, 29]             288
      BatchNorm2d-13           [-1, 32, 14, 29]              64
 Depthwise1x1Conv-14           [-1

In [72]:
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=test_loader
)

  0%|          | 0/100 [00:00<?, ?it/s]

Epoch 1/100 [Train]:   0%|          | 0/167 [00:00<?, ?it/s]

Epoch 1/100 [Val]:   0%|          | 0/44 [00:00<?, ?it/s]


Epoch 1:
  Train Loss: 2.8547, Train Acc: 16.71%
  Val Loss: 2.6680, Val Acc: 18.68%
  LR: 0.000300


Epoch 2/100 [Train]:   0%|          | 0/167 [00:00<?, ?it/s]

...

Epoch 99/100 [Val]:   0%|          | 0/44 [00:00<?, ?it/s]


Epoch 99:
  Train Loss: 0.2193, Train Acc: 94.02%
  Val Loss: 0.9379, Val Acc: 79.36%
  LR: 0.000000


Epoch 100/100 [Train]:   0%|          | 0/167 [00:00<?, ?it/s]

Epoch 100/100 [Val]:   0%|          | 0/44 [00:00<?, ?it/s]


Epoch 100:
  Train Loss: 0.2156, Train Acc: 94.23%
  Val Loss: 0.9430, Val Acc: 79.04%
  LR: 0.000000


In [76]:
model = PoseClassifier(
    n_landmarks=25,
    n_coords=3,
    seq_length=10,
    num_classes=22,
    dropout_rate=0.4
)